# SPMamba HLS-CMDS - Test & do SDRi / SI-SNRi tren tap test (Colab T4)

Notebook nay chay `test_hls.py` tren **test split source-disjoint** cua `data_hls.h5`, bao ca hai loai metric:

- **fixed** : output ch0 vs TIM (s1), ch1 vs PHOI (s2) theo dung thu tu — dung cho model train bang `FixedOrderLossWrapper`.
- **PIT** : lay hoan vi tot nhat — "best-case" tran tren.
- **SDR / SDRi (PIT)** qua `fast_bss_eval`, va **channel-accuracy** (perm == (0,1)).

Neu `fixed ~ PIT` va channel-accuracy cao => model gan kenh dung (ch0=tim, ch1=phoi). Neu `fixed << PIT` => tach duoc nhung dao kenh.

**Chuan bi:**
1. Runtime GPU (T4): Runtime -> Change runtime type -> T4 GPU.
2. Tren Drive co `data_hls.zip` (chua `data_hls.h5`) va thu muc checkpoint tu notebook train.
3. Chay lan luot Cell 1 -> 5. Cell 6 (luu ve Drive) va Cell 7 (nghe thu) tuy chon.


In [ ]:
# === Cell 1: Cau hinh duong dan + kiem tra GPU ===
from google.colab import drive
from pathlib import Path
import subprocess

drive.mount('/content/drive')

REPO_URL = "https://github.com/hoanganh0106/spmamba.git"
GIT_ROOT = "/content/spmamba"
REPO_DIR = "/content/spmamba/hls"
ENV_DIR  = "/content/spmamba-env"
MM       = "/content/bin/micromamba"

DRIVE_H5_ZIP  = "/content/drive/MyDrive/data_hls.zip"  # zip chua data_hls.h5
DRIVE_H5_PATH = ""  # optional: /content/drive/MyDrive/data_hls.h5 neu upload thang file H5
DATA_WORK_DIR = "/content/hls_h5"
LOCAL_H5      = "/content/data_hls.h5"
CONFIG_PATH   = f"{REPO_DIR}/configs/spmamba-hls-t4.yml"

# Checkpoint: tro vao THU MUC (tu lay *best*.ckpt moi nhat) hoac tro thang FILE .ckpt
CKPT_PATH = "/content/drive/MyDrive/SPMamba_checkpoints/SPMamba-HLS-HeartLung-T4"

OUT_DIR = "/content/test_hls_out"  # noi luu log + audio mau
SAVE_K  = 3                        # so mau dau tien luu audio de nghe thu

if CKPT_PATH.endswith(".ckpt"):
    RESULTS_DRIVE_DIR = str(Path(CKPT_PATH).parent / "test_results")
else:
    RESULTS_DRIVE_DIR = f"{CKPT_PATH}/test_results"

for k in ["REPO_DIR", "ENV_DIR", "DRIVE_H5_ZIP", "LOCAL_H5", "CONFIG_PATH",
          "CKPT_PATH", "OUT_DIR", "RESULTS_DRIVE_DIR"]:
    print(f"{k} = {globals()[k]}")
subprocess.run("nvidia-smi", shell=True, check=False)


Mounted at /content/drive
REPO_DIR = /content/spmamba
ENV_DIR = /content/spmamba-env
DRIVE_H5_ZIP = /content/drive/MyDrive/data_hls.zip
LOCAL_H5 = /content/data_hls.h5
CONFIG_PATH = /content/spmamba/configs/spmamba-hls-t4.yml
CKPT_PATH = /content/drive/MyDrive/SPMamba_checkpoints/SPMamba-HLS-HeartLung-T4
OUT_DIR = /content/test_hls_out
RESULTS_DRIVE_DIR = /content/drive/MyDrive/SPMamba_checkpoints/SPMamba-HLS-HeartLung-T4/test_results


CompletedProcess(args='nvidia-smi', returncode=0)

In [ ]:
# === Cell 2: Cai moi truong + thu vien (bo qua neu env da co, vd vua train xong tren cung VM) ===
import os, subprocess, shutil
from pathlib import Path

os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba-root"
os.environ["PATH"] = "/content/bin:" + os.environ.get("PATH", "")

FORCE_RECREATE_ENV = False
INSTALL_DEPS = FORCE_RECREATE_ENV or not Path(ENV_DIR).exists()  # dat True neu can cai lai

def run(cmd, cwd=None, check=True):
    print(f"\n[cmd] {cmd}", flush=True)
    return subprocess.run(cmd, shell=True, cwd=cwd, env=os.environ.copy(), text=True, check=check)

Path('/content/bin').mkdir(parents=True, exist_ok=True)
if not Path(MM).exists():
    run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content bin/micromamba")
run(f"{MM} --version")

if FORCE_RECREATE_ENV and Path(ENV_DIR).exists():
    shutil.rmtree(ENV_DIR)
if not Path(ENV_DIR).exists():
    run(f"{MM} create -p {ENV_DIR} python=3.10 --yes -q")

if INSTALL_DEPS:
    run(f"{MM} run -p {ENV_DIR} python -m pip install -U pip -q")
    run(f"{MM} run -p {ENV_DIR} pip install 'setuptools<70' typeguard==2.13.3 h5py pytorch-lightning==2.0.2 einops soundfile librosa fast-bss-eval speechbrain==0.5.14 hydra-core rich torch-complex packaging wandb -q")
    run(f"{MM} run -p {ENV_DIR} pip install --force-reinstall scipy -q")
    run(f"{MM} run -p {ENV_DIR} pip install torch-mir-eval torch-optimizer transformers -q")
    run(f"{MM} run -p {ENV_DIR} pip install torch==2.5.0 torchaudio==2.5.0 torchvision==0.20.0 --index-url https://download.pytorch.org/whl/cu124 --force-reinstall -q")
    run(f"{MM} run -p {ENV_DIR} pip install 'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.0/causal_conv1d-1.6.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl' 'https://github.com/state-spaces/mamba/releases/download/v2.3.0/mamba_ssm-2.3.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl' --no-deps --force-reinstall -q")

run(f"{MM} run -p {ENV_DIR} python -c \"import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())\"")
run(f"{MM} run -p {ENV_DIR} python -c \"from mamba_ssm.modules.mamba_simple import Mamba; print('mamba_ssm OK')\"")
run(f"{MM} run -p {ENV_DIR} python -c \"import fast_bss_eval, soundfile, h5py; print('eval deps OK')\"")
run("nvidia-smi", check=False)



[cmd] curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content bin/micromamba

[cmd] /content/bin/micromamba --version

[cmd] /content/bin/micromamba create -p /content/spmamba-env python=3.10 --yes -q

[cmd] /content/bin/micromamba run -p /content/spmamba-env python -m pip install -U pip -q

[cmd] /content/bin/micromamba run -p /content/spmamba-env pip install 'setuptools<70' typeguard==2.13.3 h5py pytorch-lightning==2.0.2 einops soundfile librosa fast-bss-eval speechbrain==0.5.14 hydra-core rich torch-complex packaging wandb -q

[cmd] /content/bin/micromamba run -p /content/spmamba-env pip install --force-reinstall scipy -q

[cmd] /content/bin/micromamba run -p /content/spmamba-env pip install torch-mir-eval torch-optimizer transformers -q

[cmd] /content/bin/micromamba run -p /content/spmamba-env pip install torch==2.5.0 torchaudio==2.5.0 torchvision==0.20.0 --index-url https://download.pytorch.org/whl/cu124 --force-reinstall -q

[cmd] /content/bin/micr

CompletedProcess(args='nvidia-smi', returncode=0)

In [ ]:
# === Cell 3: Clone/cap nhat repo ===
from pathlib import Path
import subprocess, os

def run_list(cmd, cwd=None, check=True):
    print("\n[cmd]", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, env=os.environ.copy(), text=True, check=check)

repo = Path(REPO_DIR)
if repo.exists():
    run_list(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    run_list(["git", "clone", REPO_URL, REPO_DIR])
run_list(["git", "-C", REPO_DIR, "log", "--oneline", "-3"], check=False)

if not (Path(REPO_DIR) / "test_hls.py").exists():
    raise FileNotFoundError("Repo chua co test_hls.py - can push main len GitHub roi chay lai cell nay.")



[cmd] git clone https://github.com/hoanganh0106/spmamba.git /content/spmamba

[cmd] git -C /content/spmamba log --oneline -3


In [ ]:
# === Cell 4: Giai nen/copy H5 da build san ve SSD local ===
from pathlib import Path
import h5py, shutil, zipfile

REFRESH_LOCAL_H5 = False
work_dir = Path(DATA_WORK_DIR)
work_dir.mkdir(parents=True, exist_ok=True)

if REFRESH_LOCAL_H5 and Path(LOCAL_H5).exists():
    Path(LOCAL_H5).unlink()

if not Path(LOCAL_H5).exists():
    if DRIVE_H5_PATH and Path(DRIVE_H5_PATH).exists():
        print("[data] copy H5 tu Drive:", DRIVE_H5_PATH)
        shutil.copy2(DRIVE_H5_PATH, LOCAL_H5)
    elif DRIVE_H5_ZIP and Path(DRIVE_H5_ZIP).exists():
        print("[data] giai nen H5 tu zip:", DRIVE_H5_ZIP)
        with zipfile.ZipFile(DRIVE_H5_ZIP) as zf:
            h5_members = [name for name in zf.namelist() if name.lower().endswith((".h5", ".hdf5")) and not name.endswith("/")]
            if not h5_members:
                raise RuntimeError(f"Zip khong co file .h5/.hdf5: {DRIVE_H5_ZIP}")
            if len(h5_members) > 1:
                print("[data] zip co nhieu H5, dung file dau tien:", h5_members)
            member = h5_members[0]
            extracted = work_dir / Path(member).name
            with zf.open(member) as src, open(extracted, "wb") as dst:
                shutil.copyfileobj(src, dst)
        shutil.copy2(extracted, LOCAL_H5)
        print("[data] extracted:", extracted)
    else:
        raise FileNotFoundError(f"Khong thay H5 zip/file tren Drive. Sua DRIVE_H5_ZIP hoac DRIVE_H5_PATH. DRIVE_H5_ZIP={DRIVE_H5_ZIP}")
else:
    print("[data] da co", LOCAL_H5)

with h5py.File(LOCAL_H5, "r") as f:
    print("[h5 attrs]", dict(f.attrs))
    first_id = f["mixture_ids"][0].decode("utf-8") if isinstance(f["mixture_ids"][0], bytes) else f["mixture_ids"][0]
    g = f["mixtures"][first_id]
    print("[h5 first]", first_id, g["mix"].shape, g["s1"].shape, g["s2"].shape, "has_s3=", "s3" in g)


[data] giai nen H5 tu zip: /content/drive/MyDrive/data_hls.zip
[data] extracted: /content/hls_h5/data_hls.h5
[h5 attrs] {'created_at': '2026-07-03T13:52:36.519363+00:00', 'mixing': 'dynamic_hs_ls', 'n_heart_sources': np.int64(50), 'n_lung_sources': np.int64(50), 'n_samples': np.int64(1000), 'n_src': np.int64(2), 'peak': np.float64(0.9), 'rms_norm': np.True_, 'sample_rate': np.int64(4000), 'seed': np.int64(42), 'snr_max_db': np.float64(5.0), 'snr_min_db': np.float64(-5.0), 'split_mode': 'source_disjoint', 'target_length': np.int64(60000)}
[h5 first] M00001 (60000,) (60000,) (60000,) has_s3= False


In [ ]:
# === Cell 5: TEST - do fixed/PIT SI-SNR(i) + SDR(i) tren tap test ===
import os, subprocess
from pathlib import Path

if not Path(LOCAL_H5).exists():
    raise RuntimeError("Chua co /content/data_hls.h5. Chay Cell 4 truoc.")
if not Path(CKPT_PATH).exists():
    raise RuntimeError(f"Khong thay checkpoint: {CKPT_PATH}. Kiem tra lai CKPT_PATH o Cell 1.")

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

cmd = [MM, "run", "-p", ENV_DIR, "python", "-u", "test_hls.py",
       "--ckpt", CKPT_PATH, "--h5", LOCAL_H5, "--conf_dir", CONFIG_PATH,
       "--outdir", OUT_DIR, "--save_k", str(SAVE_K)]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONPATH"] = REPO_DIR

print("[test] lenh:", " ".join(cmd))
log_lines = []
proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
    log_lines.append(line)
proc.wait()
print(f"\n[test] ket thuc - exit code {proc.returncode}")

log_path = Path(OUT_DIR) / "test_log.txt"
log_path.write_text("".join(log_lines))
print("[test] log da luu:", log_path)


[test] lenh: /content/bin/micromamba run -p /content/spmamba-env python -u test_hls.py --ckpt /content/drive/MyDrive/SPMamba_checkpoints/SPMamba-HLS-HeartLung-T4 --h5 /content/data_hls.h5 --conf_dir /content/spmamba/configs/spmamba-hls-t4.yml --outdir /content/test_hls_out --save_k 3
[auto] dùng checkpoint: /content/drive/MyDrive/SPMamba_checkpoints/SPMamba-HLS-HeartLung-T4/epoch=31-best.ckpt
/content/spmamba/test_evaluate.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this m

In [ ]:
# === Cell 6 (tuy chon): Luu log + audio mau ve Drive ===
import shutil
from pathlib import Path

src = Path(OUT_DIR)
dst = Path(RESULTS_DRIVE_DIR)
dst.mkdir(parents=True, exist_ok=True)

n = 0
for p in sorted(src.glob("*")):
    shutil.copy2(p, dst / p.name)
    n += 1
print(f"[drive] da copy {n} file -> {dst}")


[drive] da copy 16 file -> /content/drive/MyDrive/SPMamba_checkpoints/SPMamba-HLS-HeartLung-T4/test_results


In [ ]:
# === Cell 7 (tuy chon): Nghe thu cac mau da tach (mix / est / ref) ===
from pathlib import Path
from IPython.display import Audio, HTML, display

wavs = sorted(Path(OUT_DIR).glob("*_mix.wav"))
if not wavs:
    print("Chua co audio mau - chay Cell 5 truoc (SAVE_K > 0).")

for mixp in wavs:
    name = mixp.name[:-len("_mix.wav")]
    display(HTML(f"<h4>Mau {name}</h4>"))
    for suffix, label in [("_mix.wav",           "Mixture"),
                          ("_est_ch0_heart.wav", "Est ch0 - TIM"),
                          ("_ref_heart.wav",     "Ref TIM"),
                          ("_est_ch1_lung.wav",  "Est ch1 - PHOI"),
                          ("_ref_lung.wav",      "Ref PHOI")]:
        p = mixp.with_name(name + suffix)
        if p.exists():
            display(HTML(f"<b>{label}</b>"))
            display(Audio(str(p)))
